In [ ]:
# Portfolio market risk (multi-asset)
# pip install -e ".[data,models,viz]" from repo root
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import market_risk as mr
from market_risk.data.types import PortfolioPaths
from market_risk.models.garch import fit_dcc_with_fallback
from market_risk import viz

Path("images").mkdir(exist_ok=True)
paths = PortfolioPaths()
ALPHA = mr.ALPHA
EST_WINDOW = 252


In [ ]:
# Optional: refresh data (requires MASSIVE_API_KEY)
# !ingest-market-data --lookback 720 --fred-backup


In [ ]:
panel = mr.build_portfolio_panel(paths.stocks_csv, paths.bonds_csv)
pf_logrets = panel.equal_weight_portfolio()
return_dates = panel.dates
print(panel.n_days, "days,", panel.n_assets, "assets")


In [ ]:
viz.portfolio_eda_figure(pf_logrets, return_dates)
plt.show()
viz.portfolio_distribution_figure(pf_logrets, "images/portfolio-return-distributions.png")
plt.show()


In [ ]:
# In-sample VaR / ES
rows = []
for name, fn in [("historical", mr.historical_var_es), ("normal", mr.parametric_var_es_normal), ("t", mr.parametric_var_es_t)]:
    v, e = fn(pf_logrets, ALPHA)
    rows.append({"method": name, "VaR%": mr.log_var_to_loss(v)*100, "ES%": mr.log_var_to_loss(e)*100})
pd.DataFrame(rows)


In [ ]:
log_df = panel.to_dataframe().drop(columns=["date"])
dcc = fit_dcc_with_fallback(log_df, pf_logrets, ALPHA)
for k, m in dcc.items():
    if "var_log" in m:
        print(k, mr.log_var_to_loss(m["var_log"])*100, mr.log_var_to_loss(m["es_log"])*100)


In [ ]:
bt, summary = mr.rolling_var_backtest(pf_logrets, est_window=EST_WINDOW, dates=return_dates)
summary.to_csv(paths.backtest_summary_csv, index=False)
summary


In [ ]:
import numpy as np

t_res = bt["t"]
loss_var = [mr.log_var_to_loss(x)*100 for x in t_res.var_log]
r_loss = -(np.exp(pf_logrets[EST_WINDOW:]) - 1) * 100
viz.rolling_var_figure(t_res.dates, r_loss, loss_var, save_path="images/portfolio-rolling-var-t.png")
plt.show()
